# Attention, Written Out in Numpy

**Computational Text Analysis Pack · Module 25 — Transformers and Attention**

*Technique:* Scaled dot-product attention by hand; queries, keys and values

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/12-neural-foundations/25a_attention_from_scratch.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Attention is usually introduced with a diagram and a formula. Both are quicker
than the truth, which is that the mechanism is about six lines of arithmetic.

We write it out on one short sentence, so that when module 23's pretrained
models are used as instruments you know exactly what is inside them.

## The tooling

**Keras** (on TensorFlow) for the models here — its `Sequential` API is the most
readable way to state an architecture, which is why teaching material reaches
for it. **PyTorch** is the research default and what almost every paper you read
will use; the concepts transfer directly, only the syntax differs.

Both are installed here. Nothing in these two modules needs a GPU: the models
are a few hundred thousand parameters and train on CPU in under a minute.

In [ ]:
import numpy as np

# One sentence, one small hand-made embedding space.
sentence = ["the", "minister", "visited", "the", "hospital"]
rng = np.random.default_rng(7)
D = 6
E = {w: rng.normal(0, 1, D) for w in set(sentence)}
X = np.vstack([E[w] for w in sentence])
print("input matrix X:", X.shape, "(one row per token)")

## Queries, keys, values

Each token produces three vectors by multiplying by learned matrices.

- **Query** — what this token is looking for.
- **Key** — what this token offers to others.
- **Value** — what it passes on if attended to.

A token's output is a weighted average of every token's *value*, weighted by how
well its **query** matches their **key**. That is the whole mechanism.

In [ ]:
Wq = rng.normal(0, .5, (D, D))
Wk = rng.normal(0, .5, (D, D))
Wv = rng.normal(0, .5, (D, D))

Q, K, V = X @ Wq, X @ Wk, X @ Wv
print("Q", Q.shape, "K", K.shape, "V", V.shape)

scores = Q @ K.T / np.sqrt(D)          # scaled dot product
def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)
A = softmax(scores)                     # attention weights, rows sum to 1
out = A @ V

print("\nattention weight matrix (rows sum to 1):")
print(pd.DataFrame(np.round(A, 3), index=sentence, columns=sentence).to_string())
print(f"\nrow sums: {np.round(A.sum(axis=1), 6)}")
print("output:", out.shape, "— same shape as the input, contextually mixed")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(A, cmap="viridis")
ax.set_xticks(range(len(sentence))); ax.set_xticklabels(sentence, rotation=45, ha="right")
ax.set_yticks(range(len(sentence))); ax.set_yticklabels(sentence)
ax.set_xlabel("attended to"); ax.set_ylabel("attending from")
ax.set_title("Attention weights (random, untrained)")
fig.colorbar(im, ax=ax, shrink=.8)
plt.tight_layout()
print("These weights are random — the matrices were never trained. The PATTERN")
print("is meaningless; only the mechanism is the lesson here.")

## Why this beats recurrence

Look at the matrix shape. Every token attends to every other **in one
operation** — no loop, no state passed along a chain.

- **Parallel.** All positions computed at once, which is why transformers train
  on hardware that LSTMs cannot exploit.
- **No decay.** Token 1 and token 500 are one step apart, not 499.
- **Cost.** The matrix is n x n, so doubling the sequence quadruples the work.
  That quadratic cost is why context windows are a headline number.

In [ ]:
print(f"{'sequence length':>18}{'attention cells':>18}{'vs length 64':>14}")
print("-" * 52)
for n in [64, 128, 512, 2048, 8192]:
    print(f"{n:>18,}{n*n:>18,}{n*n/(64*64):>13,.0f}x")
print()
print("This is the whole reason long-context models are hard, and the reason a")
print("DH researcher with book-length documents cannot simply feed in the book.")

## Attention is not explanation

A caution that matters more in humanities work than anywhere else, because
attention maps are visually persuasive and get published as though they showed
reasoning.

Jain and Wallace (2019) demonstrated that attention weights can be substantially
altered without changing a model's prediction — so they cannot be *the* reason
for it. Wiegreffe and Pinter (2019) replied that this depends on what
"explanation" is being claimed. The dispute is unsettled.

Treat an attention map as a description of where information flowed, not as the
model's justification. Module 15's rule-based extraction can point at the rule
that fired; a transformer cannot, and no heatmap changes that.

### 🔧 Your turn 1

Set all of Wq to zeros and re-run. What happens to the attention matrix, and what does that tell you about what queries do?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Every query becomes the zero vector, so every score is zero, softmax returns a
uniform distribution, and each token's output becomes the plain average of all
values — the model collapses to exactly the `GlobalAveragePooling1D` of module
13. That is a useful thing to have felt: attention is a *learned generalisation*
of averaging, and averaging is the degenerate case where the model has decided
every token is equally relevant.

</details>